### Places365 pin classification - EDA and preprocessing


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from torchvision import models

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data" else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
INTERIM_ROOT = DATA_ROOT / "interim"
PROCESSED_ROOT = DATA_ROOT / "processed"
for folder in (RAW_ROOT, INTERIM_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'Places365'
LOCAL_CANDIDATES = [
    '../data/raw/places365',,
'../data/raw/places365_standard',,
'../data/raw/places365_standard/val',
]
THEME = 'scene'
RANDOM_STATE = 42
MAX_CLASSES = 30
MAX_IMAGES_PER_PIN = 4
sns.set_theme(style="whitegrid")


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}


def normalize_label(label: str) -> str:
    return str(label).replace("_", " ").replace("-", " ").strip().lower()


def scan_image_folder(root: Path) -> pd.DataFrame:
    records = []
    for image_path in root.rglob("*"):
        if image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        relative_parts = image_path.relative_to(root).parts
        if len(relative_parts) < 2:
            continue
        label_name = normalize_label(relative_parts[-2])
        records.append(
            {
                "image_path": str(image_path.resolve()),
                "label_name": label_name,
                "source": str(root),
            }
        )
    if not records:
        raise FileNotFoundError(f"No images found under {root}")
    return pd.DataFrame(records)


def load_raw_dataset() -> pd.DataFrame:
    for candidate in [Path(path) for path in LOCAL_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if candidate.exists():
            if candidate.is_dir():
                return scan_image_folder(candidate)
            if candidate.suffix == ".parquet":
                frame = pd.read_parquet(candidate)
            else:
                frame = pd.read_csv(candidate)
            if "image_path" not in frame.columns:
                path_like = [col for col in frame.columns if "path" in col.lower() or "image" in col.lower()]
                frame["image_path"] = frame[path_like[0]]
            if "label_name" not in frame.columns:
                label_like = [col for col in frame.columns if "label" in col.lower() or "class" in col.lower()]
                frame["label_name"] = frame[label_like[0]].map(normalize_label)
            return frame[["image_path", "label_name"]].copy()
    raise FileNotFoundError("Local dataset was not found. Place the raw dataset under ../data/raw/.")


def build_text_fields(label_name: str, image_count: int) -> dict:
    pretty = label_name.title()
    tokens = [token for token in re.findall(r"[a-zA-Zа-яА-Я0-9]+", label_name) if token]
    theme_map = {
        "scene": "place",
        "food": "dish",
        "nature": "species",
    }
    focus = theme_map.get(THEME, "topic")
    return {
        "title": f"{pretty} pin",
        "description": f"A curated {focus} pin about {label_name} with {image_count} supporting images.",
        "tags": json.dumps(tokens[:8] + [THEME, focus], ensure_ascii=False),
        "board_name": f"{pretty} ideas",
    }


def build_pin_manifest(image_df: pd.DataFrame, images_per_pin: int = 3) -> pd.DataFrame:
    working = image_df.copy()
    class_counts = working["label_name"].value_counts()
    keep_labels = class_counts.head(MAX_CLASSES).index
    working = working[working["label_name"].isin(keep_labels)].copy()
    working = working.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    working["item_idx"] = working.groupby("label_name").cumcount()
    working["pin_chunk"] = working["item_idx"] // images_per_pin

    pins = []
    for (label_name, pin_chunk), group in working.groupby(["label_name", "pin_chunk"]):
        paths = group["image_path"].head(MAX_IMAGES_PER_PIN).tolist()
        if len(paths) < 2:
            continue
        fields = build_text_fields(label_name, len(paths))
        pins.append(
            {
                "pin_id": f"{DATASET_NAME.lower()}_{label_name.replace(' ', '_')}_{pin_chunk:05d}",
                "label_name": label_name,
                "image_paths_json": json.dumps(paths, ensure_ascii=False),
                "image_count": len(paths),
                **fields,
            }
        )
    return pd.DataFrame(pins)


raw_df = load_raw_dataset()
pin_df = build_pin_manifest(raw_df, images_per_pin=3)
pin_df["label"] = pd.factorize(pin_df["label_name"])[0]
display(pin_df.head())
print("Pins:", len(pin_df), "| classes:", pin_df["label_name"].nunique())


In [ ]:
pin_df["title_len"] = pin_df["title"].str.len()
pin_df["description_len"] = pin_df["description"].str.len()
pin_df["tags_count"] = pin_df["tags"].map(lambda x: len(json.loads(x)))


def image_sizes(paths_json: str) -> dict:
    widths, heights = [], []
    for path in json.loads(paths_json):
        with Image.open(path) as image:
            widths.append(image.width)
            heights.append(image.height)
    return {
        "mean_width": float(np.mean(widths)),
        "mean_height": float(np.mean(heights)),
        "max_width": int(np.max(widths)),
        "max_height": int(np.max(heights)),
    }


image_stats = pin_df["image_paths_json"].map(image_sizes).apply(pd.Series)
pin_df = pd.concat([pin_df, image_stats], axis=1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
top_classes = pin_df["label_name"].value_counts().head(15).rename_axis("label_name").reset_index(name="count")
sns.barplot(data=top_classes, x="count", y="label_name", ax=axes[0, 0])
sns.histplot(data=pin_df, x="image_count", bins=np.arange(1, 7) - 0.5, ax=axes[0, 1])
sns.histplot(data=pin_df, x="title_len", bins=20, ax=axes[0, 2])
sns.histplot(data=pin_df, x="description_len", bins=20, ax=axes[1, 0])
sns.scatterplot(data=pin_df.sample(min(len(pin_df), 300), random_state=42), x="mean_width", y="mean_height", hue="label_name", legend=False, ax=axes[1, 1])
sns.boxplot(data=pin_df, x="image_count", y="tags_count", ax=axes[1, 2])
plt.tight_layout()


In [ ]:
sample_row = pin_df.sample(1, random_state=7).iloc[0]
sample_paths = json.loads(sample_row["image_paths_json"])
cols = min(4, len(sample_paths))
fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
axes = np.atleast_1d(axes)
for ax, path in zip(axes, sample_paths):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.axis("off")
plt.suptitle(f"{sample_row['title']} | class={sample_row['label_name']}")
plt.tight_layout()

pin_df["text_bundle"] = (
    pin_df["title"].fillna("") + " [SEP] " +
    pin_df["description"].fillna("") + " [SEP] " +
    pin_df["board_name"].fillna("") + " [SEP] " +
    pin_df["tags"].fillna("")
)

train_df, valid_df = train_test_split(
    pin_df,
    test_size=0.2 if len(pin_df) >= 100 else 0.3,
    stratify=pin_df["label"] if pin_df["label"].nunique() > 1 else None,
    random_state=42,
)
train_df["split"] = "train"
valid_df["split"] = "valid"
pin_df = pd.concat([train_df, valid_df], ignore_index=True)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
encoder.fc = torch.nn.Identity()
encoder = encoder.to(device).eval()
preprocess = models.ResNet50_Weights.IMAGENET1K_V2.transforms()


def embed_pin(paths_json: str) -> np.ndarray:
    vectors = []
    for path in json.loads(paths_json):
        with Image.open(path) as image:
            batch = preprocess(image.convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            vectors.append(encoder(batch).detach().cpu().numpy()[0])
    return np.mean(np.stack(vectors, axis=0), axis=0)


train_img = np.vstack(train_df["image_paths_json"].map(embed_pin))
valid_img = np.vstack(valid_df["image_paths_json"].map(embed_pin))

text_vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), sublinear_tf=True)
train_txt = text_vectorizer.fit_transform(train_df["text_bundle"])
valid_txt = text_vectorizer.transform(valid_df["text_bundle"])

from scipy.sparse import hstack

train_features = hstack([train_txt, train_img])
valid_features = hstack([valid_txt, valid_img])
clf = LogisticRegression(max_iter=3000, multi_class="multinomial")
clf.fit(train_features, train_df["label"])
valid_pred = clf.predict(valid_features)
print(classification_report(valid_df["label"], valid_pred, digits=4))


In [ ]:
export_path = PROCESSED_ROOT / f"{DATASET_NAME.lower().replace('-', '_')}_pin_manifest.parquet"
pin_df.to_parquet(export_path, index=False)

profile = {
    "dataset": DATASET_NAME,
    "pins": int(len(pin_df)),
    "classes": int(pin_df["label_name"].nunique()),
    "avg_images_per_pin": float(pin_df["image_count"].mean()),
    "top_classes": pin_df["label_name"].value_counts().head(10).to_dict(),
}
profile_path = INTERIM_ROOT / f"{DATASET_NAME.lower().replace('-', '_')}_pin_profile.json"
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
profile
